In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go
import logging
import torch.optim as optim
import utils
import network

import z_utils

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


# Data Loading

In [2]:
train_loader = utils.get_m2s_loader(128)
test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')

# Hyper Parameter Searching

In [ ]:
def m2sVAE_hsearch(model_class, space, dataloader, device, model_name, num_epochs):
    best_loss = float('inf')
    best_config = {}

    # Initialize the best_config with the first values from the space
    for key in space:
        best_config[key] = space[key][0]

    for key in space:
        logging.info(f"Optimizing hyperparameter: {key}")

        best_value = None
        for value in space[key]:
            current_config = best_config.copy()
            current_config[key] = value

            # Load the medoid configuration
            with open('Config/m2s_config.json', 'r') as file:
                m2s_config = json.load(file)
            model_m2s = network.m2s_VAE(**m2s_config).to(device)

            logging.info(f"Testing configuration: {current_config}")

            model_m2s = model_class(
                encoder_hidden=current_config['encoder_hidden'],
                encoder_lstm_size=current_config['encoder_lstm_size'],
                encoder_lstm_layers=current_config['encoder_lstm_layers'],
                latent_dim=current_config['latent_dim'],
                decoder_hidden=current_config['decoder_hidden'],
                embed_size_cluster=current_config['embed_size_cluster'],
                embed_size_spike=current_config['embed_size_spike'],
                decoder_layers=current_config['decoder_layers'],
                decoder_lstm_size=current_config['decoder_lstm_size'],
                decoder_lstm_dropout=current_config['decoder_lstm_dropout'],
                time_size=current_config['time_size'],
                statistics_size=current_config['statistics_size'],
                gradient_size=current_config['gradient_size'],
                fc_dropout=current_config['fc_dropout']
            ).to(device)

            optimizer = optim.Adam(model_m2s.parameters(), lr=7e-5)
        
            # Training process
            current_loss = m2s_train_for_search(model_m2s, num_epochs, dataloader, optimizer, device)
        
            if current_loss < best_loss:
                best_loss = current_loss
                best_value = value
                best_config[key] = value

        logging.info(f'Best value for {key}: {best_value}')
        logging.info(f'Current best configuration: {best_config}')
    
    logging.info(f'Best configuration: {best_config}')
    logging.info(f'Best loss: {best_loss}')

    return best_config, best_loss

def m2s_train_for_search(model_m2s, num_epochs, dataloader, optimizer, device):
    best_loss = float('inf')

    model_m2s.train()

    patience = 0

    for epoch in range(num_epochs):
        total_loss = 0.0
        num_batches = 0

        for idx, (weather, cluster, time, statistical, spike, real_energy) in enumerate(dataloader):
            weather = weather.to(device)
            cluster = cluster.to(device).int()
            time = time.to(device)
            statistical = statistical.to(device)
            spike_type = spike[:, :, 0:1].to(dtype=torch.int32)
            spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
            spike_type = spike_type.to(device)
            spike_magnitude = spike_magnitude.to(device)
            real_energy = real_energy.to(device)
            noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)

            optimizer.zero_grad()
            synthetic_profile, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)

            loss = utils.vae_loss_function(synthetic_profile, real_energy, mu, logvar)
            loss += 10 * utils.gradient_loss(synthetic_profile, statistical)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            num_batches += 1

        epoch_loss = total_loss / num_batches

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            patience = 0
        else:
            patience += 1
        
        if patience == 3:
            break

    return best_loss

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define hyperparameter space
hyperparameter_space = {
    'encoder_hidden': [128, 256, 512, 1024], # 
    'encoder_lstm_size': [8, 16, 32],
    'encoder_lstm_layers': [2, 3, 4],
    'latent_dim': [512, 1024, 2048],
    'decoder_hidden': [512, 1024, 2048],
    'embed_size_cluster': [2, 3, 4],
    'embed_size_spike': [1, 2, 3, 4],
    'decoder_layers': [3, 4, 5],
    'decoder_lstm_size': [32, 64, 128],
    'decoder_lstm_dropout': [0.2, 0.3, 0.4],
    'time_size': [32, 64, 128],
    'statistics_size': [64, 128, 256],
    'gradient_size': [128, 256, 512, 1024],
    'fc_dropout': [0.2, 0.3],
}

# Example call
best_config, best_loss = m2sVAE_hsearch(network.m2s_VAE, hyperparameter_space, train_loader, device, 'm2s_vae_test', 20)

with open('Config/s_vae_config.json', 'w') as file:
    json.dump(best_config, file, indent=4)

2025-09-01 19:52:42,922 - INFO - Optimizing hyperparameter: encoder_hidden
2025-09-01 19:52:43,088 - INFO - Testing configuration: {'encoder_hidden': 128, 'encoder_lstm_size': 8, 'encoder_lstm_layers': 2, 'latent_dim': 512, 'decoder_hidden': 512, 'embed_size_cluster': 2, 'embed_size_spike': 1, 'decoder_layers': 3, 'decoder_lstm_size': 32, 'decoder_lstm_dropout': 0.2, 'time_size': 32, 'statistics_size': 64, 'gradient_size': 128, 'fc_dropout': 0.2}


KeyboardInterrupt: 

# Training

In [8]:
best_config_path = 'Config/s_vae_config.json'

with open(best_config_path, 'r') as file:
    best_config = json.load(file)

model_m2s = network.m2s_VAE(**best_config).to(device)
optimizer = optim.Adam(model_m2s.parameters(), lr=1e-5)

model_m2s.train()

model_m2s, best_loss = z_utils.train_s_vae(model_m2s, optimizer, train_loader, device, 2)

torch.save(model_m2s.state_dict(), '../../Models/s_vae2.pt')

2026-01-11 15:55:25,623 - INFO - Step 10000 / 44144, Loss: 64.23075866699219
2026-01-11 15:56:50,623 - INFO - Step 20000 / 44144, Loss: 54.84511947631836
2026-01-11 15:58:06,039 - INFO - Step 30000 / 44144, Loss: 64.0431137084961
2026-01-11 15:59:12,949 - INFO - Step 40000 / 44144, Loss: 54.69868469238281
2026-01-11 15:59:46,746 - INFO - Epoch 1/2, Loss: 162.70712660891976
2026-01-11 16:00:57,969 - INFO - Step 10000 / 44144, Loss: 54.552799224853516
2026-01-11 16:02:16,096 - INFO - Step 20000 / 44144, Loss: 73.52200317382812
2026-01-11 16:03:30,966 - INFO - Step 30000 / 44144, Loss: 55.814002990722656
2026-01-11 16:04:45,404 - INFO - Step 40000 / 44144, Loss: 95.0067138671875
2026-01-11 16:05:12,360 - INFO - Epoch 2/2, Loss: 69.32461168313208
2026-01-11 16:05:12,360 - INFO - Best loss achieved: 69.32461168313208


# Performance

In [5]:
best_config_path = 'Config/s_vae_config.json'

with open(best_config_path, 'r') as file:
    best_config = json.load(file)

model_m2s = network.m2s_VAE(**best_config).to(device)

model_m2s.load_state_dict(torch.load('../../Models/s_vae.pt'))

model_m2s.eval()

m2s_VAE(
  (lstm_encoder): m2s_Encoder(
    (lstm): LSTM(1, 8, num_layers=2, batch_first=True, bidirectional=True)
    (fc1): Sequential(
      (0): Linear(in_features=16, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (4): ReLU()
    )
    (fc2): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=1024, bias=True)
      (4): ReLU()
    )
  )
  (lstm_decoder): m2s_Decoder(
    (Embedding_spike): Embedding(5, 1)
    (Embedding_cluster): Embedding(20, 2)
    (lstm): LSTM(512, 32, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
    (time_net): Sequential(
      (0): Linear(in_features=4, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=32, bi

## The Training Set

In [6]:
eva_training_loader = utils.get_m2s_loader(1)
long_fake_energy = torch.Tensor().to(device)
long_real_energy = torch.Tensor().to(device)

for i in range(12):

    weather, cluster, time, statistical, spike, real_energy = next(iter(eva_training_loader)) 

    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike[:, :, 0:1].to(dtype=torch.int32)
    spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)
    real_energy = real_energy.to(device)

    noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)

    synthetic_profile, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)

    long_fake_energy = torch.cat((long_fake_energy, synthetic_profile), dim=-1)
    long_real_energy = torch.cat((long_real_energy, real_energy), dim=-1)

long_fake_energy = long_fake_energy.flatten().detach().cpu().numpy()
long_real_energy = long_real_energy.flatten().detach().cpu().numpy()

fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(long_real_energy)), y=long_real_energy, mode='lines', name='Real', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, len(long_fake_energy)), y=long_fake_energy, mode='lines', name='Synthetic', line=dict(color='orange')))

fig.show()

## The Test Set

In [ ]:

weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(48 * 3, test_df)

weather = weather.to(device)
cluster = cluster.to(device).int()
time = time.to(device)
statistical = statistical.to(device)
spike_type = spike_type.to(dtype=torch.int32)  # or torch.long for indexing
spike_magnitude = spike_magnitude.to(dtype=torch.float32)
spike_type = spike_type.to(device)
spike_magnitude = spike_magnitude.to(device)
real_energy = torch.Tensor(real_energy).to(device)

real_energy = real_energy.unsqueeze(0)

noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)

synthetic_profile, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)

synthetic_profile = synthetic_profile.flatten().detach().cpu().numpy()
real_energy = real_energy.flatten().detach().cpu().numpy()

fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(real_energy)), y=real_energy, mode='lines', name='Real', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, len(synthetic_profile)), y=synthetic_profile, mode='lines', name='Synthetic', line=dict(color='orange')))

fig.show()


# Evaluation

Similarity and Correlation with the real energy profile: MSE, Spearman's Rank Correlation Coefficient, Correlation

In [ ]:
mse = []
srcc = []
corr = []

for i in range(100):

    weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(48 * 3, test_df)

    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike_type.to(dtype=torch.int32)  # or torch.long for indexing
    spike_magnitude = spike_magnitude.to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)
    real_energy = torch.Tensor(real_energy).to(device)
    real_energy = real_energy.unsqueeze(0)

    noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)

    synthetic_profile, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)

    synthetic_profile = synthetic_profile.flatten().detach().cpu().numpy()
    real_energy = real_energy.flatten().detach().cpu().numpy()

    mse.append(utils.mean_squared_error(synthetic_profile, real_energy))
    srcc.append(utils.spearman_correlation(synthetic_profile, real_energy))
    corr.append(utils.pearson_correlation(synthetic_profile, real_energy))

print(f'MSE: {np.mean(mse)}')
print(f'SRCC: {np.mean(srcc)}')
print(f'Corr: {np.mean(corr)}')
    
    

MSE: 0.03444598242640495
SRCC: 0.4607923693569208
Corr: 0.5693339445422669
